In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotrnv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [ ]:
load_dotenv()

In [ ]:
model = ChatOpenAI(model = 'gpt-4o-mini')

In [ ]:
class EvaluationScheme(BaseModel):

    feedback: str = Field(description = "Detailed feedback for the essay")
    score : int = Field(description = "Score out of 10", ge = 0, le = 10)

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

In [ ]:
essay = "this is an essay"

In [ ]:
prompt = f'evaluate this essay for it's quality and give a feedback as well as assign a score for it'
structured_model.invoke(prompt)

In [ ]:
class UPSCState(TypedDict):

    essay : str
    language_feedback : str
    analysis_feedback : str
    clarity_feedback : str
    overall_feedback : str
    individual_scores : Annotated[list[int], operator.add]
    avg_score : float

In [ ]:
def evaluate_language(state: UPSCState):
    prompt = f'Evaluate the language quality of the following essay {state['essay']} give feedback on it and also assign score based on your understanding'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores' : [output.score]}
    
    

In [ ]:
def evaluate_analysis(state: UPSCState):

    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

def evaluate_thought(state: UPSCState):

    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

def final_evaluation(state: UPSCState):

    # summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content

    # avg calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}
    

In [ ]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evalaute_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evalute_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evalute_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [ ]:
workflow